In [3]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, cross_val_score

# ── Load the data ──────────────────────────────────────────────────────────────
df = pd.read_csv("1632349boston.csv")

print("=" * 70)
print("ASSIGNMENT 3: MACHINE LEARNING - PREDICTING MEDIAN HOUSE PRICE (MEDV)")
print("=" * 70)

# ── Step 1: Prepare the data (one-hot encoding) ───────────────────────────────
# From Class 4 slide 6: in machine learning, dummy variables are called
# "one-hot coding" — same process as in regression

df_model = pd.get_dummies(df, columns=['CHAS', 'LANDMARK'], dtype=int)
df_model.columns = df_model.columns.str.replace(' ', '_')
df_model = df_model.drop(columns=['CHAS_no', 'LANDMARK_No_Landmark'])

print("\nFeatures used for prediction:")
features = [c for c in df_model.columns if c != 'MEDV']
print(features)

# ── Step 2: Define features (X) and target (y) ────────────────────────────────
# From Class 4 slide 6:
# Independent Variable = Feature  →  X
# Dependent Variable   = Target   →  y

X = df_model[features]
y = df_model['MEDV']

# ── Step 3: Set up 5-fold cross validation ────────────────────────────────────
# From Class 4 slide 14 & 15: KFold with 5 splits, shuffle the data
# random_state ensures the result is reproducible

kf = KFold(n_splits=5, shuffle=True, random_state=42)

# ── Step 4: Run linear regression with cross_val_score ────────────────────────
# From Class 4 slide 15 (exact class code):
# cross_val_score runs the model on each fold automatically
# scoring='neg_mean_absolute_error' gives the MAD/MAE per fold (negative)

lr = LinearRegression()

scores = cross_val_score(lr, X, y,
                         scoring='neg_mean_absolute_error',
                         cv=kf)

# cross_val_score returns negative values so we multiply by -1
MAD_scores = scores * -1

# ── Step 5: Print results ─────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("RESULTS: LINEAR REGRESSION — 5-FOLD CROSS VALIDATION")
print("=" * 70)

for i, mad in enumerate(MAD_scores, 1):
    print(f"Fold {i}: MAD = {mad:.2f} ($1,000s)  →  ${mad * 1000:,.0f}")

print(f"\nAverage MAD across all 5 folds : {MAD_scores.mean():.2f} ($1,000s)")
print(f"Average MAD across all 5 folds : ${MAD_scores.mean() * 1000:,.0f}")



ASSIGNMENT 3: MACHINE LEARNING - PREDICTING MEDIAN HOUSE PRICE (MEDV)

Features used for prediction:
['CRIM', 'INDUS', 'NOX', 'RM', 'AGE', 'DIS', 'RAD', 'TAX', 'PTRATIO', 'CHAS_yes', 'LANDMARK_Museum', 'LANDMARK_Park', 'LANDMARK_Shopping_Mall', 'LANDMARK_Stadium']

RESULTS: LINEAR REGRESSION — 5-FOLD CROSS VALIDATION
Fold 1: MAD = 45.97 ($1,000s)  →  $45,967
Fold 2: MAD = 46.48 ($1,000s)  →  $46,483
Fold 3: MAD = 51.61 ($1,000s)  →  $51,613
Fold 4: MAD = 47.82 ($1,000s)  →  $47,819
Fold 5: MAD = 56.67 ($1,000s)  →  $56,666

Average MAD across all 5 folds : 49.71 ($1,000s)
Average MAD across all 5 folds : $49,710


In [ ]:
#EXPLANATION OF RESULTS

# RESULTS PER FOLD:
# - Fold 1: MAD = $45,967  — the model was off by $45,967 on average
# - Fold 2: MAD = $46,483  — the model was off by $46,483 on average
# - Fold 3: MAD = $51,613  — slightly worse fold, more difficult data to predict
# - Fold 4: MAD = $47,819  — the model was off by $47,819 on average
# - Fold 5: MAD = $56,666  — the worst fold, highest prediction error
# - The variation between folds is normal — each fold has different data points

# AVERAGE MAD = $49,710 — THIS IS OUR BENCHMARK:
# - On average, the linear regression model predicts house prices with an
#   error of $49,710.
# - Remember: MEDV is in $1,000s, so a MAD of 49.71 = $49,710 in real money.
# - This number is our benchmark — any more advanced model we try next
#   should aim to get a lower MAD than $49,710 to be considered better.

# WHAT IS K-FOLD CROSS VALIDATION? (Class 4 slide 14):
# - The data was split into 5 equal parts (folds).
# - Each fold took a turn being the test data while the other 4 were
#   used for training. This means every data point got tested exactly once.
# - We then averaged the MAD across all 5 folds for a reliable estimate.
# - This is better than just splitting the data once, because it ensures
#   the result is not just lucky or unlucky with one particular split.

# HOW TO EXPLAIN THIS TO A FRIEND (non-technical summary):
# - We taught the computer to predict house prices using all the
#    neighbourhood characteristics in the data.
# - To test how accurate it is, we split the data into 5 parts and
#    tested it 5 times — each time with a different test set.
# - On average, the model's prediction was off by about $49,710
#    per house — that is our starting benchmark.
# - In the next steps, we will try more advanced techniques to see
#    if we can get that error number lower than $49,710."

In [5]:
import pandas as pd
import numpy as np
from sklearn.linear_model import Lasso, Ridge
from sklearn.model_selection import KFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import MinMaxScaler

# ── Load the data ──────────────────────────────────────────────────────────────
df = pd.read_csv("1632349boston.csv")

print("=" * 70)
print("ASSIGNMENT 3 Q2: LASSO & RIDGE WITH HYPERPARAMETER TUNING")
print("=" * 70)

# ── Step 1: One-hot encoding (same as Q1) ─────────────────────────────────────
df_model = pd.get_dummies(df, columns=['CHAS', 'LANDMARK'], dtype=int)
df_model.columns = df_model.columns.str.replace(' ', '_')
df_model = df_model.drop(columns=['CHAS_no', 'LANDMARK_No_Landmark'])

# ── Step 2: Define features (X) and target (y) ────────────────────────────────
features = [c for c in df_model.columns if c != 'MEDV']
X = df_model[features]
y = df_model['MEDV']

# ── Step 3: Normalize X to range 0-1 ─────────────────────────────────────────
# From Class 4 slide 25: Lasso and Ridge require variables on the same scale
# Normalization transforms each variable to range from 0 to 1
# Exact code from slide 25:

columns = X.columns                          # save column names
scaler  = MinMaxScaler()                     # initialise the scaler
scaler.fit_transform(X)                      # fit the scaler
X       = scaler.fit_transform(X)            # scale the data
X       = pd.DataFrame(X, columns=columns)  # put back into a dataframe

print("\nData normalized to range 0-1 (required for Lasso and Ridge)")

# ── Step 4: Set up 5-fold cross validation (same as Q1) ──────────────────────
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# ── Step 5: Grid search for optimal alpha — LASSO ─────────────────────────────
# From Class 5 slide 5: GridSearchCV goes over all alpha values you specify
# and returns the one that gives the best prediction
# We use a wide range of alpha values as shown in class

print("\n" + "=" * 70)
print("GRID SEARCH: Finding the optimal alpha for LASSO")
print("=" * 70)

# Define the range of alpha values to try (wide range as shown in class)
alpha_range = np.arange(0.1, 100, 0.5)

# Define the model and the parameter grid (exact class code style)
lasso_model = Lasso()
param_lasso  = {'alpha': alpha_range}

# Run grid search with 5-fold cross validation
lasso_grid = GridSearchCV(lasso_model,
                          param_lasso,
                          scoring='neg_mean_absolute_error',
                          cv=kf)
lasso_grid.fit(X, y)

best_alpha_lasso = lasso_grid.best_params_['alpha']
print(f"Best alpha for Lasso : {round(best_alpha_lasso, 1)}")

# ── Step 6: Grid search for optimal alpha — RIDGE ─────────────────────────────
print("\n" + "=" * 70)
print("GRID SEARCH: Finding the optimal alpha for RIDGE")
print("=" * 70)

ridge_model = Ridge()
param_ridge  = {'alpha': alpha_range}

ridge_grid = GridSearchCV(ridge_model,
                          param_ridge,
                          scoring='neg_mean_absolute_error',
                          cv=kf)
ridge_grid.fit(X, y)

best_alpha_ridge = ridge_grid.best_params_['alpha']
print(f"Best alpha for Ridge : {round(best_alpha_ridge, 1)}")

# ── Step 7: Run final models with optimal alpha using cross_val_score ──────────
# From Class 4 slide 26: use cross_val_score with the best alpha values
# (same structure as Q1 linear regression)

print("\n" + "=" * 70)
print("FINAL MAD WITH OPTIMAL ALPHA VALUES")
print("=" * 70)

# Lasso with best alpha
lasso_final  = Lasso(alpha=best_alpha_lasso)
lasso_scores = cross_val_score(lasso_final, X, y,
                               scoring='neg_mean_absolute_error',
                               cv=kf)
lasso_mad = lasso_scores.mean() * -1

# Ridge with best alpha
ridge_final  = Ridge(alpha=best_alpha_ridge)
ridge_scores = cross_val_score(ridge_final, X, y,
                               scoring='neg_mean_absolute_error',
                               cv=kf)
ridge_mad = ridge_scores.mean() * -1

print(f"Lasso  (alpha = {round(best_alpha_lasso, 1)}): MAD = {lasso_mad:.2f} ($1,000s)  →  ${lasso_mad * 1000:,.0f}")
print(f"Ridge  (alpha = {round(best_alpha_ridge, 1)}): MAD = {ridge_mad:.2f} ($1,000s)  →  ${ridge_mad * 1000:,.0f}")

# ── Step 8: Summary comparison ────────────────────────────────────────────────
print("\n" + "=" * 70)
print("SUMMARY: COMPARISON OF ALL MODELS")
print("=" * 70)
print(f"Linear Regression (benchmark) : MAD = 49.71 ($1,000s)  →  $49,710")
print(f"Lasso (alpha = {round(best_alpha_lasso, 1):<6})        : MAD = {lasso_mad:.2f} ($1,000s)  →  ${lasso_mad * 1000:,.0f}")
print(f"Ridge (alpha = {round(best_alpha_ridge, 1):<6})        : MAD = {ridge_mad:.2f} ($1,000s)  →  ${ridge_mad * 1000:,.0f}")

best_model = "Lasso" if lasso_mad < ridge_mad else "Ridge"
print(f"\n→ {best_model} performs better than the other for this dataset.")



ASSIGNMENT 3 Q2: LASSO & RIDGE WITH HYPERPARAMETER TUNING

Data normalized to range 0-1 (required for Lasso and Ridge)

GRID SEARCH: Finding the optimal alpha for LASSO
Best alpha for Lasso : 0.1

GRID SEARCH: Finding the optimal alpha for RIDGE
Best alpha for Ridge : 0.6

FINAL MAD WITH OPTIMAL ALPHA VALUES
Lasso  (alpha = 0.1): MAD = 49.61 ($1,000s)  →  $49,611
Ridge  (alpha = 0.6): MAD = 49.28 ($1,000s)  →  $49,285

SUMMARY: COMPARISON OF ALL MODELS
Linear Regression (benchmark) : MAD = 49.71 ($1,000s)  →  $49,710
Lasso (alpha = 0.1   )        : MAD = 49.61 ($1,000s)  →  $49,611
Ridge (alpha = 0.6   )        : MAD = 49.28 ($1,000s)  →  $49,285

→ Ridge performs better than the other for this dataset.


In [ ]:
#EXPLANATION OF RESULTS

# HYPERPARAMETER TUNING RESULTS:
# - Grid search tried alpha values from 0.1 to 100 automatically.
# - Best alpha for Lasso : 0.1  (very small penalty)
# - Best alpha for Ridge : 0.6  (also a small penalty)
# - Both alphas being small makes sense for this dataset — the variables
#   are already quite informative, so a strong penalty would hurt more
#   than it helps.

# COMPARISON OF ALL THREE MODELS:
# - Linear Regression (benchmark) : MAD = $49,710
# - Lasso  (alpha = 0.1)          : MAD = $49,611  → $99 better than benchmark
# - Ridge  (alpha = 0.6)          : MAD = $49,285  → $425 better than benchmark
# - Ridge is the best performing model of the three.

# WHAT DO THESE RESULTS MEAN?
# - All three models perform very similarly on this dataset.
# - Neither Lasso nor Ridge improved dramatically over linear regression.
# - This tells us that overfitting was not a big problem in the
#   linear regression model to begin with — the penalty correction
#   of Lasso and Ridge did not help much.
# - Ridge slightly outperforms Lasso, meaning it found a better
#   balance between fitting the training data and predicting new data.

# HOW TO EXPLAIN THIS TO A FRIEND (non-technical summary):
# - We tried two smarter prediction models called Lasso and Ridge
#    that include a built-in correction to avoid overcomplicating things.
# - We used an automatic search to find the best setting for each model
#    instead of guessing — this is called hyperparameter tuning.
# - The results show that Ridge is the best model so far, with an
#    average prediction error of $49,285.
# - However, the improvement over our original model is very small —
#    only about $425 — which means the simple linear regression was
#    already doing a pretty good job.
# - In the next step we will try a neural network to see if we can
#    get a much bigger improvement.

In [8]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold, cross_val_score
from sklearn.preprocessing import MinMaxScaler
from tensorflow import keras
from tensorflow.keras import layers
from scikeras.wrappers import KerasRegressor

# ── Load the data ──────────────────────────────────────────────────────────────
df = pd.read_csv("1632349boston.csv")

print("=" * 70)
print("ASSIGNMENT 3 Q3: NEURAL NETWORK - PREDICTING MEDIAN HOUSE PRICE")
print("=" * 70)

# ── Step 1: One-hot encoding (same as Q1 and Q2) ──────────────────────────────
df_model = pd.get_dummies(df, columns=['CHAS', 'LANDMARK'], dtype=int)
df_model.columns = df_model.columns.str.replace(' ', '_')
df_model = df_model.drop(columns=['CHAS_no', 'LANDMARK_No_Landmark'])

# ── Step 2: Define features (X) and target (y) ────────────────────────────────
features = [c for c in df_model.columns if c != 'MEDV']
X = df_model[features]
y = df_model['MEDV']

# ── Step 3: Normalize X to range 0-1 (same as Q2) ────────────────────────────
# From Class 4 slide 25: normalize so all variables are on the same scale

columns = X.columns
scaler  = MinMaxScaler()
X       = scaler.fit_transform(X)
X       = pd.DataFrame(X, columns=columns)

print("\nData normalized to range 0-1")
print(f"Number of features : {X.shape[1]}")

# ── Step 4: Define the neural network model ───────────────────────────────────
# From Class 5 slide 15: define a function that creates the neural network
# Hyperparameters specified in the assignment:
#   - 512 nodes
#   - 4 hidden layers
#   - 100 epochs
#   - 16 batch size
#   - relu activation function
#   - Adam optimizer
#   - MAD (mean_absolute_error) loss function

def create_baseline():
    # create model
    model = keras.Sequential()

    # Input layer
    model.add(layers.Input(shape=(X.shape[1],)))

    # 4 hidden layers with 512 nodes and relu activation (from assignment)
    model.add(layers.Dense(512, activation='relu'))   # hidden layer 1
    model.add(layers.Dense(512, activation='relu'))   # hidden layer 2
    model.add(layers.Dense(512, activation='relu'))   # hidden layer 3
    model.add(layers.Dense(512, activation='relu'))   # hidden layer 4

    # Output layer — 1 node for regression (predicting house price)
    model.add(layers.Dense(1))

    # Compile with Adam optimizer and MAD (mean_absolute_error) loss
    # From assignment: Adam optimizer, MAD loss function
    model.compile(loss='mean_absolute_error',
                  optimizer='adam',
                  metrics=['mean_absolute_error'])  # MAD to evaluate the model

    return model

# ── Step 5: Wrap model for sklearn (from Class 5 slide 16) ───────────────────
# KerasRegressor allows us to use cross_val_score with a keras model
# epochs = 100, batch_size = 16 (from assignment)

estimator = KerasRegressor(model=create_baseline,
                           epochs=100,
                           batch_size=16,
                           verbose=0)  # verbose=0 silences training output

# ── Step 6: Run 5-fold cross validation ──────────────────────────────────────
# From Class 5 slide 16: same cross_val_score approach as Q1 and Q2
# random_state=42 to keep results reproducible

kf = KFold(n_splits=5, shuffle=True, random_state=42)

print("\n" + "=" * 70)
print("RUNNING NEURAL NETWORK WITH 5-FOLD CROSS VALIDATION")
print("(This may take a few minutes...)")
print("=" * 70)
print("Hyperparameters:")
print("  Nodes per layer : 512")
print("  Hidden layers   : 4")
print("  Epochs          : 100")
print("  Batch size      : 16")
print("  Activation      : relu")
print("  Optimizer       : Adam")
print("  Loss function   : mean_absolute_error (MAD)")

results = cross_val_score(estimator, X, y,
                          cv=kf,
                          scoring='neg_mean_absolute_error')

MAD_scores = results * -1

# ── Step 7: Print results ─────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("RESULTS: NEURAL NETWORK — 5-FOLD CROSS VALIDATION")
print("=" * 70)

for i, mad in enumerate(MAD_scores, 1):
    print(f"Fold {i}: MAD = {mad:.2f} ($1,000s)  →  ${mad * 1000:,.0f}")

nn_mad = MAD_scores.mean()
print(f"\nAverage MAD : {nn_mad:.2f} ($1,000s)  →  ${nn_mad * 1000:,.0f}")

# ── Step 8: Final comparison of all models ────────────────────────────────────
print("\n" + "=" * 70)
print("FINAL SUMMARY: COMPARISON OF ALL MODELS")
print("=" * 70)
print(f"Linear Regression (benchmark) : MAD = 49.71 ($1,000s)  →  $49,710")
print(f"Lasso  (alpha = 0.1)          : MAD = 49.61 ($1,000s)  →  $49,611")
print(f"Ridge  (alpha = 0.6)          : MAD = 49.28 ($1,000s)  →  $49,285")
print(f"Neural Network                : MAD = {nn_mad:.2f} ($1,000s)  →  ${nn_mad * 1000:,.0f}")

if nn_mad < 49.28:
    print("\n→ The neural network beats all other models!")
else:
    print("\n→ The neural network does not improve over Ridge regression.")



I0000 00:00:1774599754.485774   51944 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1774599754.487148   51944 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1774599754.540511   51944 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1774599758.006260   51944 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.

ASSIGNMENT 3 Q3: NEURAL NETWORK - PREDICTING MEDIAN HOUSE PRICE

Data normalized to range 0-1
Number of features : 14

RUNNING NEURAL NETWORK WITH 5-FOLD CROSS VALIDATION
(This may take a few minutes...)
Hyperparameters:
  Nodes per layer : 512
  Hidden layers   : 4
  Epochs          : 100
  Batch size      : 16
  Activation      : relu
  Optimizer       : Adam
  Loss function   : mean_absolute_error (MAD)


E0000 00:00:1774599760.308079   51944 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)



RESULTS: NEURAL NETWORK — 5-FOLD CROSS VALIDATION
Fold 1: MAD = 35.41 ($1,000s)  →  $35,415
Fold 2: MAD = 43.42 ($1,000s)  →  $43,425
Fold 3: MAD = 43.59 ($1,000s)  →  $43,590
Fold 4: MAD = 38.76 ($1,000s)  →  $38,762
Fold 5: MAD = 46.25 ($1,000s)  →  $46,245

Average MAD : 41.49 ($1,000s)  →  $41,487

FINAL SUMMARY: COMPARISON OF ALL MODELS
Linear Regression (benchmark) : MAD = 49.71 ($1,000s)  →  $49,710
Lasso  (alpha = 0.1)          : MAD = 49.61 ($1,000s)  →  $49,611
Ridge  (alpha = 0.6)          : MAD = 49.28 ($1,000s)  →  $49,285
Neural Network                : MAD = 41.49 ($1,000s)  →  $41,487

→ The neural network beats all other models!


In [ ]:
#EXPLANATION OF RESULTS 


# RESULTS PER FOLD:
# - Fold 1: $35,415  — the neural network predicted best in this fold
# - Fold 2: $43,425
# - Fold 3: $43,590
# - Fold 4: $38,762
# - Fold 5: $46,245  — the neural network was furthest off in this fold
# - The variation between folds is normal — each fold tests on
#   slightly different data so the error naturally varies.

# FINAL RESULT:
# - Average MAD = $41,487
# - This means the neural network is on average off by $41,487
#   when predicting a median house price.

# COMPARISON OF ALL MODELS:
# - Linear Regression (benchmark) : MAD = $49,710
# - Lasso  (alpha = 0.1)          : MAD = $49,611  → $99 better than benchmark
# - Ridge  (alpha = 0.6)          : MAD = $49,285  → $425 better than benchmark
# - Neural Network                : MAD = $41,487  → $8,223 better than benchmark
# - The neural network is clearly the best performing model.
# - It reduced the prediction error by about $8,223 compared to
#   linear regression — a much bigger improvement than Lasso or Ridge.

# HOW TO EXPLAIN THIS TO A FRIEND (non-technical summary):
# - The neural network is like a much smarter version of our
#    previous models — it has 4 layers of 512 neurons each that
#    can detect complex patterns in the data.
# - We tested it 5 times with different parts of the data to
#    make sure the result is reliable.
# - On average it was off by $41,487 — that is $8,223 better
#    than our original linear regression model.
# - In other words: if a house is really worth $300,000, the
#    neural network would predict somewhere around $258,000 to
#    $342,000 — much closer than before.
# - This shows that for predicting house prices, a neural network
#    is clearly the best tool out of the three we tried.